### config

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.connect.functions import to_date, unix_timestamp
from pyspark.sql.functions import column

import config.ConnectionConfig as cc
cc.setupEnvironment()

spark = cc.startLocalCluster("fact_rides",7)
spark.getActiveSession()

25/05/09 12:13:40 WARN Utils: Your hostname, 4L3KS-comp resolves to a loopback address: 127.0.1.1; using 10.140.99.119 instead (on interface wlp2s0)
25/05/09 12:13:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/aleks/Downloads/bigtools/spark-3.5.4-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/aleks/.ivy2/cache
The jars for the packages stored in: /home/aleks/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
org.elasticsearch#elasticsearch-spark-30_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2e97f4ac-6532-44c5-9120-02f0587dfc04;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.

# EXTRACT

I tried using the spark API first, but there are some limitations on their joining of columns.

In [2]:
# EXTRACT rides:
# rides_table_SQL = '(SELECT * FROM rides) as rides_table'
# df_rides = spark.read.format("jdbc")\
#     .option("driver" , cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", rides_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "rideid") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_rides.printSchema()
#df_rides.show()
#df_rides.count()
# ----

#EXTRACT bike_type through bike_lot through vehicle:
# Rename to avoid duplicate column names
# Corrected column name
# vehicle_table_SQL = """
# (
#     SELECT
#         v.*,
#         l.bikelotid AS bikelotid_bikelots,
#         l.deliverydate,
#         l.biketypeid
#     FROM vehicles v
#     LEFT JOIN bikelots l ON v.bikelotid = l.bikelotid
# ) AS vehicle_table
# """
#
# df_vehicles = spark.read.format("jdbc")\
#     .option("driver", cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", vehicle_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "vehicleid") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_vehicles.show()
# ------

#EXTRACT users through subscriptionid through subscription
# user_table_SQL = """
# (
#     SELECT
#         userid AS userid_subscriptions,
#         subscriptionid AS subscriptionid_subscriptions
#     FROM subscriptions s
# ) AS user_table
# """
#
# df_users = spark.read.format("jdbc")\
#     .option("driver", cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", user_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "userid_subscriptions") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_users.show()

# ------

#EXTRACT date
# df_dates = spark.read.format("delta").load("../dimensions/spark-warehouse/dimdate")
# df_dates.show()
# df_dates.createOrReplaceTempView("dates")

# ------

#EXTRACT weather: TBD.



Trying to do it in one SQL query:

In [2]:
# EXTRACTING ALL IN ONE GO:
# load dates from deltatable:
df_dates = spark.read.format("delta").load("../dimensions/spark-warehouse/dimdate")
#df_dates.show()
df_dates.createOrReplaceTempView("dates")

# write the SQL string needed:
SQL = """(
    SELECT
    r.*,
    v.vehicleid AS vehicleid_vehicles,
    v.bikelotid AS bikelotid_vehicles,
    b.bikelotid AS bikelotid_bikelots,
    b.biketypeid AS biketypeid_bikelots,
    s.subscriptionid AS subscriptionid_subscriptions,
    s.userid AS userid_subscriptions
        FROM rides r
            LEFT JOIN vehicles v ON r.vehicleid = v.vehicleid
            LEFT JOIN bikelots b ON v.bikelotid = b.bikelotid
            LEFT JOIN subscriptions s ON r.subscriptionid = s.subscriptionid

) as rides_table
"""

df_rides_full = spark.read.format("jdbc")\
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", SQL) \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0)\
    .option("upperBound", 100) \
    .load()

df_rides_full.show()
# WEATHER would be done separately


+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+------------------+------------------+------------------+-------------------+----------------------------+--------------------+
|rideid|       startpoint|         endpoint|          starttime|            endtime|vehicleid|subscriptionid|startlockid|endlockid|vehicleid_vehicles|bikelotid_vehicles|bikelotid_bikelots|biketypeid_bikelots|subscriptionid_subscriptions|userid_subscriptions|
+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+------------------+------------------+------------------+-------------------+----------------------------+--------------------+
|     1|(51.2083,4.44595)|(51.1938,4.40228)|2015-09-22 00:00:00|2012-09-22 00:00:00|      844|         13296|       4849|     3188|               844|                 3|                 3|                  1|               

In [3]:
#EXTRACT
df_users = spark.read.load('./spark-warehouse/dim_user/dim_user.snappy.parquet')
df_users.createOrReplaceTempView("users_dim")
df_dates = spark.read.load('./spark-warehouse/dimdate/dim_date.snappy.parquet')
df_dates.createOrReplaceTempView("dates_dim")
df_users.show()

+-------+-------+--------------------+--------+-------+--------------------+------------+----------+----------+--------------+----------+------------------+---------------------+-------+
|user_sk|user_id|              street|  number|zipcode|                city|country_code|start_date|  end_date|subscriptionid| validfrom|subscriptiontypeid|end_subscription_date|current|
+-------+-------+--------------------+--------+-------+--------------------+------------+----------+----------+--------------+----------+------------------+---------------------+-------+
|      0|      6|   Jan Ockegemstraat|168 0107|   2650|              Edegem|          BE|2019-06-29|2020-06-28|             8|2019-06-29|                 3|           2020-06-28|   true|
|      1|      6|   Jan Ockegemstraat|168 0107|   2650|              Edegem|          BE|2023-11-30|2024-11-29|             9|2023-11-30|                 3|           2024-11-29|  false|
|      2|      6|   Jan Ockegemstraat|168 0107|   2650|          

# TRANSFORM

In [12]:
#TRANSFORMATION
date_joined_df = df_users.join(df_rides_full,
        (df_users.subscriptionid == df_rides_full.subscriptionid_subscriptions) &
        (df_users.validfrom  <= df_rides_full.starttime) &
        (df_users.end_subscription_date >= df_rides_full.starttime),"inner")

joined_df = (
    date_joined_df
    .join(df_dates, df_dates.CalendarDate == date_joined_df.starttime.cast('date'), "leftouter")
    .select(
        df_dates.dateSK.alias("dates_sk"),
        df_rides_full.rideid,
        df_rides_full.startlockid.alias("start_lock_id"),
        df_rides_full.endlockid.alias("end_lock_id"),
        df_rides_full.vehicleid.alias("vehicle_id"),
        df_users.user_sk.alias("user_sk"),
        (df_rides_full.endtime.cast("long") - df_rides_full.starttime.cast("long")).alias("duration") #this is in seconds, turn into minutes probs
    )
)
joined_df.show()


+--------+------+-------------+-----------+----------+-------+--------+
|dates_sk|rideid|start_lock_id|end_lock_id|vehicle_id|user_sk|duration|
+--------+------+-------------+-----------+----------+-------+--------+
|       2|    15|         4849|       3188|       844|   6655|     893|
|       2|    16|         NULL|       NULL|      4545|  23057|     124|
|       2|    18|         1821|       2186|      1208|  15450|     304|
|       2|    22|         2572|         13|      2015|  32680|     335|
|       2|    23|           50|       2067|      5298|  35667|     395|
|       2|    25|          985|       2148|      1400|    510|     271|
|       2|    26|         2039|       3038|       957|  30033|     641|
|       2|    27|         5619|       2717|      5413|   2561|    1176|
|       2|    28|         3531|       3554|      2658|  33017|       0|
|       2|    29|         4940|       1218|      1925|  19276|     194|
|       2|    32|         1056|        792|      3572|  14820|  

# LOAD

In [13]:
joined_df.repartition(1).write.format("parquet").mode("overwrite").saveAsTable('temp_fact_parquet')

In [14]:
spark.stop()